1. Loading Dependencies

In [ ]:
import os
import sys
import time
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader

In [ ]:
# -------------------------------------------------
# Add project root to Python path
# -------------------------------------------------

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
# -------------------------------------------------
# Preprocessing
# -------------------------------------------------
from src.preprocessing.image_processing import (
    preprocess_image
)

from src.preprocessing.spectrogram import (
    preprocess_audio
)

# -------------------------------------------------
# Dataset processing
# -------------------------------------------------

from src.datasets.label_processing import (
    prepare_binary_labels
)

from src.datasets.multimodal_dataset import (
    MultimodalDataset
)

# -------------------------------------------------
# Models
# -------------------------------------------------
from src.models.semantic_alignment import (
    SharedConvNet,
    ClassifierNet
)
# from src.models.fusion import FeatureFusionModel
# from src.models.baselines import VisualOnlyCNN
# from src.models.baselines import AudioOnlyCNN

# -------------------------------------------------
# Training
# -------------------------------------------------
from src.training.trainer import (
    Trainer
)

# -------------------------------------------------
# Evaluation
# -------------------------------------------------
from src.evaluation.metrics import evaluate_model

# -------------------------------------------------
# Explainability
# -------------------------------------------------
from src.explainability.lime_analysis import run_lime

2. Loading Raw Data

In [ ]:
# -------------------------------------------------
# Dataset paths
# -------------------------------------------------
VISUAL_FOLDER = "../data/raw/images"

AUDIO_FOLDER = "../data/raw/audio_files"

LABEL_PATH = "../data/raw/labels/annotations_1.csv"

# -------------------------------------------------
# Output paths
# -------------------------------------------------

OUTPUT_FOLDER = "../outputs"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

In [ ]:
# -------------------------------------------------
# Load Raw Data
# -------------------------------------------------

visual_files = sorted(os.listdir(VISUAL_FOLDER))
audio_files = sorted(os.listdir(AUDIO_FOLDER))

print("Number of visual files:", len(visual_files))
print("Number of audio files:", len(audio_files))

In [ ]:
# -------------------------------------------------
# Load metadata
# -------------------------------------------------

metadata = pd.read_csv(
    LABEL_PATH
)

print(metadata.head())

print(metadata.shape)

In [ ]:
# -------------------------------------------------
# Prepare binary labels
# -------------------------------------------------

metadata, labels = prepare_binary_labels(
    metadata
)

# -------------------------------------------------
# Verify labels
# -------------------------------------------------

print("Metadata shape:")

print(metadata.shape)

print("Unique labels:")

print(np.unique(labels))

print("NaN labels:")

print(np.isnan(labels).sum())

In [ ]:
# -------------------------------------------------
# Get valid filenames
# -------------------------------------------------

image_filenames = metadata[
    "image_file_name"
].values

audio_filenames = metadata[
    "audio_file_name"
].values

print("Number of image files:")

print(len(image_filenames))

print("Number of audio files:")

print(len(audio_filenames))

3. Data Preprocessing

In [ ]:
# -------------------------------------------------
# Preprocess Image Data
# -------------------------------------------------

# -------------------------------------------------
# Create image dataset container
# -------------------------------------------------

im_dataset = np.zeros(
    (
        len(image_filenames),
        80,
        80
    )
)

image_sample_record = []

# -------------------------------------------------
# Start timer
# -------------------------------------------------

t0 = time.time()

# -------------------------------------------------
# Preprocess images
# -------------------------------------------------

for i in range(len(image_filenames)):

    # print(
    #     f"Processing image {i+1}/{len(image_filenames)}"
    # )

    image_path = os.path.join(
        VISUAL_FOLDER,
        image_filenames[i]
    )

    processed_image = preprocess_image(
        image_path
    )

    image_sample_record.append([
        image_filenames[i],
        processed_image
    ])

    im_dataset[i] = processed_image

print(
    "Image preprocessing time:",
    time.time() - t0
)

print(im_dataset.shape)

In [ ]:
# -------------------------------------------------
# Preprocess Audio Data
# -------------------------------------------------

# -------------------------------------------------
# Create spectrogram container
# -------------------------------------------------

spectrograms = np.zeros(
    (
        len(audio_filenames),
        80,
        80
    )
)

audio_sample_record = []

# -------------------------------------------------
# Generate spectrograms
# -------------------------------------------------

for i in range(len(audio_filenames)):

    # print(
    #     f"Processing audio {i+1}/{len(audio_filenames)}"
    # )

    wav_path = os.path.join(
        AUDIO_FOLDER,
        audio_filenames[i]
    )

    processed_spec = preprocess_audio(
        wav_path
    )

    audio_sample_record.append([
        audio_filenames[i],
        processed_spec
    ])

    spectrograms[i] = processed_spec

print(spectrograms.shape)

In [ ]:
# -------------------------------------------------
# Standardize modality variable names
# -------------------------------------------------

visual_data = im_dataset

audio_data = spectrograms

print("Visual data shape:")

print(visual_data.shape)

print("Audio data shape:")

print(audio_data.shape)

In [ ]:
# -------------------------------------------------
# Train/Validation/Test Split
# -------------------------------------------------

Xv_train, Xv_temp, Xa_train, Xa_temp, y_train, y_temp = train_test_split(
    visual_data,
    audio_data,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

Xv_val, Xv_test, Xa_val, Xa_test, y_val, y_test = train_test_split(
    Xv_temp,
    Xa_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# -------------------------------------------------
# Print dataset shapes
# -------------------------------------------------

print("Training visual data:")

print(Xv_train.shape)

print("Training audio data:")

print(Xa_train.shape)

print("Validation visual data:")

print(Xv_val.shape)

print("Validation audio data:")

print(Xa_val.shape)

print("Testing visual data:")

print(Xv_test.shape)

print("Testing audio data:")

print(Xa_test.shape)

In [ ]:
# -------------------------------------------------
# Create PyTorch datasets
# -------------------------------------------------

train_dataset = MultimodalDataset(
    Xv_train,
    Xa_train,
    y_train
)

val_dataset = MultimodalDataset(
    Xv_val,
    Xa_val,
    y_val
)

test_dataset = MultimodalDataset(
    Xv_test,
    Xa_test,
    y_test
)

In [ ]:
# -------------------------------------------------
# Create DataLoaders
# -------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
# -------------------------------------------------
# Inspect one sample
# -------------------------------------------------

sample = train_dataset[0]

print(sample.keys())

print("Visual tensor shape:")

print(sample["visual"].shape)

print("Audio tensor shape:")

print(sample["audio"].shape)

print("Label:")

print(sample["label"])

4. Model Training

In [ ]:
# -------------------------------------------------
# Define computation device
# -------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:")

print(DEVICE)

In [ ]:
# -------------------------------------------------
# Training hyperparameters
# -------------------------------------------------

EPOCHS = 1200

LEARNING_RATE = 0.000484674265378098

WEIGHT_DECAY = 0.000793891463437805

MARGIN = 1.0

GAMMA = 0.5

print("Epochs:", EPOCHS)

print("Learning rate:", LEARNING_RATE)

print("Weight decay:", WEIGHT_DECAY)

print("Margin:", MARGIN)

print("Gamma:", GAMMA)

In [ ]:
# -------------------------------------------------
# Initialize shared feature extractor
# -------------------------------------------------

encoder_net = SharedConvNet(
    num_conv_layers=2,
    filters1=30,
    filters2=32,
    kernel_size=3
)

encoder_net = encoder_net.to(
    DEVICE
)

print(encoder_net)

In [ ]:
# -------------------------------------------------
# Initialize shared classifier
# -------------------------------------------------

task_net = ClassifierNet(
    dense_units1=220,
    output_dims=[93],
    drops=0.0752054726602688,
    filters2=32
)

task_net = task_net.to(
    DEVICE
)

print(task_net)

In [ ]:
# -------------------------------------------------
# Initialize optimizers
# -------------------------------------------------

encoder_optimizer = torch.optim.Adam(
    encoder_net.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

classifier_optimizer = torch.optim.Adam(
    task_net.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
# -------------------------------------------------
# Initialize semantic alignment trainer
# -------------------------------------------------

trainer = Trainer(
    encoder_net=encoder_net,
    task_net=task_net,
    encoder_optimizer=encoder_optimizer,
    classifier_optimizer=classifier_optimizer,
    device=DEVICE,
    margin=MARGIN,
    gamma=GAMMA
)

In [ ]:
# -------------------------------------------------
# Temporary source/target setup
# -------------------------------------------------

source_loader = train_loader

In [ ]:
# -------------------------------------------------
# Train semantic alignment framework
# -------------------------------------------------

history = trainer.fit(
    train_loader=train_loader,
    epochs=EPOCHS
)

In [ ]:
# -------------------------------------------------
# Plot training losses (skip first epoch)
# -------------------------------------------------

epochs = range(
    2,
    EPOCHS + 1
)

plt.figure(figsize=(10, 6))

# ----------------------------------------------
# L_CCSA
# ----------------------------------------------

plt.plot(
    epochs,
    history["encoder_loss"][1:],
    label=r"$L_{CCSA}$"
)

# ----------------------------------------------
# L_CSA
# ----------------------------------------------

plt.plot(
    epochs,
    history["alignment_loss"][1:],
    label=r"$L_{CSA}$"
)

# ----------------------------------------------
# L_C
# ----------------------------------------------

plt.plot(
    epochs,
    history["task_loss"][1:],
    label=r"$L_{C}$"
)

plt.plot(
    epochs,
    history["accuracy"][1:],
    label="Training Accuracy"
)

# ----------------------------------------------
# Figure settings
# ----------------------------------------------

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title(
    "Cross-Modal Semantic Alignment Training Losses"
)

plt.legend()

plt.grid(True)

plt.show()

5. Model Evaluation

In [ ]:
# -------------------------------------------------
# Evaluate visual-only deployment performance
# -------------------------------------------------

results = evaluate_model(
    encoder_net=encoder_net,
    task_net=task_net,
    dataloader=test_loader,
    device=DEVICE
)

6. LIME Explainability

In [ ]:
# -------------------------------------------------
# Select sample from test dataset
# -------------------------------------------------

sample = test_dataset[0]

visual_sample = sample["visual"]

label = sample["label"]

print("Ground truth label:")

print(label)

In [ ]:
# -------------------------------------------------
# Visualize melt pool image
# -------------------------------------------------

plt.figure(figsize=(5,5))

plt.imshow(
    visual_sample.squeeze(),
    cmap="gray"
)

plt.title(
    "Original Melt Pool Image"
)

plt.axis("off")

plt.show()

In [ ]:
# -------------------------------------------------
# Run LIME explanation
# -------------------------------------------------

explanation = run_lime(
    encoder_net=encoder_net,
    task_net=task_net,
    image=visual_sample,
    device=DEVICE,
    num_samples=1000
)